<div style="display:flex; align-items:center; gap:18px; text-align:left">
  <img src="https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/assets/logo_ust.png" width="100">
  <div>
    <p>Diplomado en Inteligencia Artificial para los Negocios</p>
    <p>Facultad de Ingeniería y Negocios</p>
    <p>Módulo 2: Fundamentos de Machine Learning y herramientas Low Code</p>
    <p>Semana 06: Ejercicios de repaso 2 · Clase 6 (solución comentada)</p>
  </div>
</div>

# Ejercicios de repaso 2 · Clase 6: Tuning, interpretación y persistencia — Solución comentada

Este notebook resuelve los ejercicios de repaso 2 de la **Clase 6**. Cada ejercicio trae:

1. **El código de la solución**, con comentarios línea a línea redactados de forma clara y directa.
2. **Por qué se hizo así**: la razón de cada decisión, la lectura esperada del resultado y el enlace directo
   a la diapositiva correspondiente del curso para profundizar.
3. En los ejercicios clave, un **prompt para pedirle a una IA que les ayude a corregir su código**.
   Está pensado para que la IA **revise lo que ustedes escribieron** y explique el error, no para que
   escriba todo de nuevo.

Úsenlo **después** de intentar cada ejercicio: comparar su intento con la solución es lo que más enseña.

## Preparación del entorno

Instalamos las librerías necesarias y dejamos la configuración general de importaciones.

In [1]:
%%capture
!pip install -q pycaret==4.0.0a8 plotly ipywidgets shap "setuptools<81"

In [2]:
%matplotlib inline
import os, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.express as px

# en Colab los datos se leen del repo público; en local, desde ../datasets/
REPO = 'https://raw.githubusercontent.com/sebastiancontz/ust-fundamentos-machine-learning-colab/main/ediciones/2026/datasets/'
BASE = '../datasets/' if os.path.exists('../datasets') else REPO

## Bloque A · Fuga de datos y evaluación honesta

**Caso:** campaña de depósito a plazo de un banco (`banco_marketing.csv`, datos reales de UCI Bank Marketing).
El banco quiere decidir **a qué clientes llamar** en la próxima campaña. La columna `contrata` vale **1**
si el cliente contrató el depósito y **0** si no.

En este bloque aprendemos a distinguir la evaluación honesta de la ilusión de un modelo con trampa.

In [3]:
banco = pd.read_csv(BASE + 'banco_marketing.csv')
print('Filas y columnas:', banco.shape)
banco.head()

Filas y columnas: (8000, 10)


,edad,educacion,credito_hipotecario,credito_consumo,duracion_llamada_seg,contactos_esta_campana,contactos_previos,resultado_campana_anterior,tasa_euribor,contrata
0,56.0,4.0,0.0,1.0,307.0,1.0,0.0,0.0,4.857,0
1,59.0,5.0,0.0,0.0,139.0,1.0,0.0,0.0,4.857,0
2,41.0,NaN,0.0,0.0,217.0,1.0,0.0,0.0,4.857,0
3,57.0,1.0,1.0,0.0,293.0,1.0,0.0,0.0,4.857,0
4,46.0,2.0,1.0,1.0,440.0,1.0,0.0,0.0,4.857,0


### Ejercicio A1 · La trampa de la variable que sabe la respuesta

Entrenen un clasificador rápido `rf` (*Random Forest*) que incluya **todas** las columnas de la tabla
(incluida `duracion_llamada_seg`). Observen su métrica **AUC** y la **importancia de variables**.
Luego respondan: ¿por qué este resultado es un engaño?, ¿conocemos la duración de la llamada el próximo
lunes a las 9:00 AM cuando el ejecutivo decide a quién marcar?

In [4]:
from pycaret.tasks import ClassificationExperiment

# probamos primero con la columna tramposa para ver el engaño
exp_trampa = ClassificationExperiment(target='contrata', session_id=42, train_size=0.7, fold=3).fit(banco)
rf_trampa = exp_trampa.create_model('rf')

print('AUC con trampa:', rf_trampa.metrics['AUC'].values[0])
exp_trampa.plot_model(rf_trampa.pipeline, plot='feature')

AUC con trampa: 0.9428


**Por qué se hizo así**

- **`duracion_llamada_seg`** dura muchos segundos si la persona se interesó y contrató, y dura pocos
  segundos si colgó de inmediato. El modelo descubre esa relación obvia y logra un **AUC sobre 0,94**.
- **Por qué es fuga de datos (*data leakage* tipo 2):** esa información **no existe** antes de hacer la
  llamada. Para armar la lista de contactos del call center, esa columna está en blanco.
- **La lección:** ningún software detecta esta fuga automáticamente. PyCaret no sabe qué significa
  "segundos de llamada". Es el **criterio humano** del analista el que debe preguntarse columna por
  columna: *¿este dato estará disponible en el momento exacto de tomar la decisión?*
- Como la respuesta es no, la columna **se descarta** para construir el modelo honesto.

Para profundizar en la distinción entre variables predictivas legítimas y variables que revelan el resultado, revisen la diapositiva [Fuga tipo 2: variables que saben la respuesta](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/06-tuning-interpretacion.html#/fuga-tipo-2-variables-que-saben-la-respuesta) de la Clase 6 (disponible a continuación de forma interactiva):

In [5]:
# diapositiva interactiva de la clase
from IPython.display import IFrame
IFrame('https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/06-tuning-interpretacion.html#/fuga-tipo-2-variables-que-saben-la-respuesta', width='100%', height=480)

**Si su código no funciona: prompt para que una IA les ayude a corregirlo**

Copien el prompt, reemplacen lo que está entre corchetes y péguenlo en su asistente de IA. Pidan una corrección explicada, no una solución nueva: el objetivo es entender qué falló.

```text
Estoy aprendiendo machine learning con PyCaret 4.0.0a8 en Google Colab sobre un caso bancario.
Entrené un modelo para predecir si un cliente contrata un depósito, pero incluí la columna `duracion_llamada_seg`.
El modelo dio un AUC de 0.94 y la duración domina toda la importancia.

Este es mi código:
[pegar aquí su código]

Explícame en lenguaje simple y orientado a negocios por qué los profesores dicen que este modelo "hace trampa"
(fuga de datos) y por qué en la práctica real fallaría por completo el lunes por la mañana en el call center.
```

### Ejercicio A2 · El experimento honesto y el pipeline como maleta

Descarten la columna `duracion_llamada_seg` de la tabla (`df = banco.drop(columns='duracion_llamada_seg')`).
Creen el experimento honesto con `train_size=0.7`, `fold=5`, `normalize=True` y `fold_strategy='stratifiedkfold'`.
Entrenen un modelo base `rf` y muestren su objeto `.pipeline`. ¿Qué transformaciones viajan dentro de esa "maleta"?
¿Por qué deben ajustarse **solo** con los datos de entrenamiento?

In [6]:
# descartamos la duración: el lunes a las 9 AM nadie sabe cuánto durará la llamada
df_honesto = banco.drop(columns='duracion_llamada_seg')

exp = ClassificationExperiment(
    target='contrata',
    session_id=42,
    train_size=0.7,
    fold=5,
    normalize=True,                  # estandarización calculada solo con datos de entrenamiento
    fold_strategy='stratifiedkfold', # cada pliegue mantiene el 11% de depósitos
).fit(df_honesto)

base = exp.create_model('rf')
modelo_base = base.pipeline          # el estimador real entrenado vive en .pipeline

print('AUC honesto en validación cruzada:', base.metrics['AUC'].values[0])
modelo_base

AUC honesto en validación cruzada: 0.7204


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocess', ...), ('rf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['edad','educacion','credito_hipotecario',...,'contactos_previos', 'resultado_campana_anterior','tasa_euribor']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numerical_pipeline', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that featu

**Por qué se hizo así**

- **El AUC baja de ~0,94 a ~0,71:** no es que el modelo empeoró; es que ahora vemos la **verdad**.
  Predecir si alguien contratará usando solo su edad, historial previo y tasas es difícil, y 0,71 es un
  desempeño respetable y honesto.
- **La maleta (`Pipeline`):** al imprimir `modelo_base`, verán que contiene las etapas de imputación de
  faltantes, codificación y escalado (`normalize=True`), seguidas por el clasificador `RandomForestClassifier`.
- **Por qué evita la fuga tipo 1:** el escalador calcula la media y la desviación estándar **únicamente**
  con el 70% de entrenamiento. El 30% de prueba queda intacto (holdout). Si escaláramos la tabla completa
  antes de separar, los datos de prueba ya habrían contaminado el entrenamiento.
- **`create_model` vs `.pipeline`:** recuerden que en PyCaret 4 `create_model` devuelve un contenedor
  `CreateResult`. El modelo real que podemos usar para predecir o graficar vive en su atributo `.pipeline`.

Para repasar cómo el pipeline actúa como salvaguarda aislando las transformaciones dentro del entrenamiento, consulten la diapositiva [El pipeline: todo el proceso en un solo paquete](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/06-tuning-interpretacion.html#/el-pipeline-todo-el-proceso-en-un-solo-paquete) de la Clase 6.

## Bloque B · Selección de variables con criterio y con método

Tener muchas columnas no siempre ayuda: variables irrelevantes agregan ruido y encarecen los sistemas.
En este bloque aprendemos a medir el aporte real de cada variable y a preguntarnos **cuántas variables bastan**.

### Ejercicio B1 · Importancia por permutación (PFI): la medida honesta

Calculen la **importancia por permutación** del modelo honesto sobre el conjunto de prueba (`exp.X_test` e
`exp.y_test`) usando `permutation_importance` de scikit-learn con scoring `'roc_auc'`.
¿Cuál es la variable que más pesa en la predicción honesta? ¿Por qué desordenar una columna al azar nos
permite saber si realmente importa?

In [7]:
from sklearn.inspection import permutation_importance

# barajamos cada columna al azar en los datos de prueba para medir cuánto cae el AUC
pfi = permutation_importance(modelo_base, exp.X_test, exp.y_test,
                             scoring='roc_auc', n_repeats=5, random_state=42)

s_pfi = pd.Series(pfi.importances_mean, index=exp.X_test.columns).sort_values(ascending=False)
print('Caída promedio del AUC al desordenar cada variable (PFI):')
display(s_pfi.round(4))

Caída promedio del AUC al desordenar cada variable (PFI):


tasa_euribor                  0.1006
resultado_campana_anterior    0.0157
edad                          0.0030
contactos_esta_campana        0.0027
contactos_previos             0.0013
credito_consumo              -0.0019
educacion                    -0.0023
credito_hipotecario          -0.0073
dtype: float64

**Por qué se hizo así**

- **Cómo funciona la permutación:** toma una columna (por ejemplo `tasa_euribor`) y **baraja sus valores
  al azar** entre los clientes. Eso destruye por completo su relación con la contratación. Si el AUC del
  modelo **se desploma**, significa que el modelo dependía críticamente de esa variable. Si el AUC casi no
  cambia, la variable era irrelevante o redundante.
- **Lectura esperada:** `tasa_euribor` provoca una caída de más de **0,10 de AUC**; es por lejos la variable
  más determinante. Le sigue `resultado_campana_anterior`.
- **Por qué es más honesta que la importancia de fábrica del árbol:** la importancia interna del árbol
  cuenta cuántas veces se divide una variable en el entrenamiento y favorece a variables con muchos valores
  distintos aunque sean puro sobreajuste. La permutación sobre datos de prueba mide el impacto real en la
  generalización.

Para revisar el fundamento matemático y las ventajas del método de permutación frente a la importancia por impureza, revisen la diapositiva [Permutación: evaluar el aporte sobre el error](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/06-tuning-interpretacion.html#/permutación-evaluar-el-aporte-sobre-el-error) de la Clase 6 (disponible a continuación de forma interactiva):

In [8]:
# diapositiva interactiva de la clase
from IPython.display import IFrame
IFrame('https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/06-tuning-interpretacion.html#/permutación-evaluar-el-aporte-sobre-el-error', width='100%', height=480)

**Si su código no funciona: prompt para que una IA les ayude a corregirlo**

Copien el prompt, reemplacen lo que está entre corchetes y péguenlo en su asistente de IA. Pidan una corrección explicada, no una solución nueva: el objetivo es entender qué falló.

```text
Estoy usando scikit-learn y PyCaret 4.0.0a8 en Colab.
Tengo un pipeline `modelo_base` y los conjuntos `exp.X_test` e `exp.y_test`.
Quiero calcular `permutation_importance` usando scoring='roc_auc' y mostrar las variables ordenadas de mayor a menor.

Este es mi código:
[pegar aquí su código]

Explícame qué está fallando y cómo interpretar el número que entrega cada variable (¿qué significa que una variable tenga 0.10 y otra 0.001?).
```

### Ejercicio B2 · ¿Cuántas variables necesita realmente el modelo?

Esta celda **ya viene provista**: ejecuta RFECV (eliminación recursiva de variables con validación
cruzada) para ver cómo evoluciona el AUC a medida que agregamos o quitamos variables.
Ejecútenla y respondan: ¿con cuántas variables se alcanza prácticamente el mejor desempeño?
Si una variable agrega apenas 0,002 de AUC pero cuesta millones capturarla, ¿la conservarían?

In [9]:
from sklearn.base import clone
from sklearn.feature_selection import RFECV

# pasamos los datos por el preprocesamiento del pipeline
preproc = modelo_base[:-1]
estimador_puro = clone(modelo_base[-1])
X_train_trans = preproc.transform(exp.X_train)

# rfecv va quitando variables de a una para ver cuándo deja de mejorar el modelo
rfecv = RFECV(estimator=estimador_puro, step=1, cv=3, scoring='roc_auc')
rfecv.fit(X_train_trans, exp.y_train)

res_rfecv = pd.DataFrame({
    'Variables conservadas': range(1, len(rfecv.cv_results_['mean_test_score']) + 1),
    'AUC (validación cruzada)': rfecv.cv_results_['mean_test_score'].round(4)
})
display(res_rfecv)
print('Número óptimo según el algoritmo:', rfecv.n_features_)

,Variables conservadas,AUC (validación cruzada)
0,1,0.7321
1,2,0.7139
2,3,0.7207
3,4,0.7440
4,5,0.7397
5,6,0.7538
6,7,0.7456
7,8,0.7432


Número óptimo según el algoritmo: 6


**Por qué se hizo así**

- **Qué hace RFECV:** parte con las 8 variables, evalúa con validación cruzada, elimina la menos
  útil, reentrena y repite. Así responde la pregunta: *¿cuántas variables hacen falta realmente?*
- **Lectura esperada:** con una sola variable (`tasa_euribor`) el modelo ya supera 0,65 de AUC. Con 4 a 6
  variables alcanza prácticamente su techo (~0,71). Las últimas 2 variables aportan casi nada.
- **Decisión de negocio:** menos variables significan un modelo **más simple, más barato de operar y más
  fácil de explicar**. El algoritmo sugiere 6 o 7 variables por milésimas técnicas, pero la decisión de
  negocio puede descartar variables costosas o difíciles de conseguir si su aporte es marginal.

Para profundizar en el trade-off entre simplicidad operativa y ganancia métrica marginal, consulten la diapositiva [¿Cuántas variables necesita nuestro modelo?](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/06-tuning-interpretacion.html#/cuántas-variables-necesita-nuestro-modelo) de la Clase 6.

## Bloque C · Hiperparámetros y tuning

Un **hiperparámetro** es una perilla que fijamos **antes** de entrenar (como la profundidad del árbol
`max_depth` o el número de estimadores `n_estimators`). El *tuning* busca de forma sistemática la mejor
combinación. En este bloque vemos cómo afinar el modelo y cómo evaluar si la mejora es real.

### Ejercicio C1 · Afinar el modelo con tune_model

Apliquen `tune_model` sobre `modelo_base` realizando 15 iteraciones con `optimize='AUC'`.
Muestren los mejores hiperparámetros encontrados (`mejora.best_params`) y exploren las primeras filas
de la tabla de búsqueda `pd.DataFrame(mejora.cv_results)`. ¿Cuánto mejoró el AUC promedio en la validación
cruzada? ¿El tuning transformó el modelo o solo afinó décimas?

In [10]:
# probamos 15 combinaciones al azar buscando maximizar el AUC
mejora = exp.tune_model(modelo_base, n_iter=15, optimize='AUC')

print('Mejores hiperparámetros encontrados:')
for k, v in mejora.best_params.items():
    print(f'  {k}: {v}')

intentos = pd.DataFrame(mejora.cv_results)
columnas_clave = [c for c in intentos.columns if c.startswith('param_')] + ['mean_test_score']
intentos_resumen = intentos[columnas_clave].sort_values('mean_test_score', ascending=False)
display(intentos_resumen.head(5))

auc_base_cv = base.metrics['AUC'].values[0]
auc_tun_cv = intentos_resumen['mean_test_score'].iloc[0]
print(f'AUC en validación cruzada: Base = {auc_base_cv:.4f} -> Afinado = {auc_tun_cv:.4f} (Ganancia: {auc_tun_cv - auc_base_cv:+.4f})')

Mejores hiperparámetros encontrados:
  n_estimators: 60
  min_samples_split: 2
  min_samples_leaf: 6
  min_impurity_decrease: 0.001
  max_features: 1.0
  max_depth: 4
  criterion: gini
  class_weight: balanced
  bootstrap: True


,param_n_estimators,param_min_samples_split,param_min_samples_leaf,param_min_impurity_decrease,param_max_features,param_max_depth,param_criterion,param_class_weight,param_bootstrap,mean_test_score
11,60,2,6,0.001,1.0,4,gini,balanced,True,0.769543
8,120,5,6,0.000,sqrt,9,gini,{},True,0.758423
14,20,10,2,0.010,log2,6,entropy,balanced_subsample,True,0.756642
7,230,10,6,0.000,sqrt,9,entropy,{},True,0.756154
6,120,2,3,0.010,log2,4,entropy,balanced_subsample,True,0.754810


AUC en validación cruzada: Base = 0.7204 -> Afinado = 0.7695 (Ganancia: +0.0491)


**Por qué se hizo así**

- **`optimize='AUC'`:** por defecto PyCaret optimiza por *Accuracy*. Con clases desbalanceadas (solo el
  11% contrata), optimizar por accuracy elegiría un modelo que predice siempre "no". Por eso es
  obligatorio especificar `optimize='AUC'`.
- **Búsqueda al azar (`RandomizedSearchCV`):** `tune_model` prueba 15 combinaciones al azar dentro de un
  espacio razonable de hiperparámetros. En `mejora.cv_results` quedan registrados todos los intentos.
- **Expectativa realista:** el tuning sube el AUC en un par de centésimas (de ~0,71 a ~0,74). El tuning
  **afina décimas, no hace milagros**. La calidad de los datos y las variables correctas mueven mucho más la
  aguja que las perillas del algoritmo.

Para repasar los espacios de búsqueda y los riesgos de optimizar métricas inadecuadas, revisen la diapositiva [Tuning: buscar la mejor combinación](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/06-tuning-interpretacion.html#/tuning-buscar-la-mejor-combinación) de la Clase 6.

**Si su código no funciona: prompt para que una IA les ayude a corregirlo**

Copien el prompt, reemplacen lo que está entre corchetes y péguenlo en su asistente de IA. Pidan una corrección explicada, no una solución nueva: el objetivo es entender qué falló.

```text
Estoy usando PyCaret 4.0.0a8 en Google Colab.
Tengo un modelo base `modelo_base` y ejecuté `mejora = exp.tune_model(modelo_base, n_iter=15, optimize='AUC')`.
Quiero ver los mejores parámetros y la tabla de todos los intentos ordenados por su score.

Este es mi código:
[pegar aquí su código]

Revisa mi código y explícame por qué es crucial pasar `optimize='AUC'` cuando el target está desbalanceado.
```

### Ejercicio C2 · El examen único en el holdout: Base vs Afinado

Evalúen `modelo_base` y `mejora.pipeline` sobre el **conjunto de prueba** (`predict_model`).
Pongan lado a lado el AUC de ambos en una tabla. ¿El modelo afinado confirma su ventaja con datos nuevos
que nunca vio? ¿Por qué esta comparación en el holdout se realiza **una sola vez** al final del proceso?

In [11]:
# examen final sobre los clientes de prueba que nadie tocó durante el ajuste
res_base = exp.predict_model(modelo_base)
res_afinado = exp.predict_model(mejora.pipeline)

comparacion = pd.DataFrame({
    'Modelo': ['Base (Random Forest)', 'Afinado (Tuned RF)'],
    'AUC Holdout': [res_base.metrics['AUC'].iloc[0], res_afinado.metrics['AUC'].iloc[0]],
    'F1 Holdout': [res_base.metrics['F1'].iloc[0], res_afinado.metrics['F1'].iloc[0]]
})
display(comparacion.round(4))

,Modelo,AUC Holdout,F1 Holdout
0,Base (Random Forest),0.7081,0.2680
1,Afinado (Tuned RF),0.7517,0.4064


**Por qué se hizo así**

- **El holdout es el árbitro final:** el tuning se hizo usando validación cruzada dentro del 70% de
  entrenamiento. El 30% de prueba no participó en ninguna decisión.
- **Lectura esperada:** el modelo afinado mantiene un AUC superior en prueba (~0,74 vs ~0,71). Esto
  confirma que la ganancia es **generalizable** y no fue una memorización accidental del tuning.
- **Por qué se usa una sola vez:** si probáramos diez estrategias de tuning y eligiéramos la que mejor
  puntúa en el holdout, estaríamos usando el holdout para entrenar. El holdout se "gastaría" y perdería su
  honestidad como examen imparcial.

Para entender por qué el conjunto de prueba se utiliza estrictamente una sola vez como auditor final, consulten la diapositiva [Secuencia: evaluar, finalizar y persistir](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/06-tuning-interpretacion.html#/secuencia-evaluar-finalizar-y-persistir) de la Clase 6.

## Bloque D · Interpretabilidad con SHAP y persistencia

Los modelos modernos son complejos. Para generar confianza, cumplir con regulaciones (como la Ley 21.719
en Chile) y detectar sesgos, debemos **abrir la caja negra**: explicar qué variables pesan en general
(**global**) y por qué se tomó una decisión sobre una persona en particular (**local**).
Luego, empaquetamos el modelo para que el banco pueda usarlo en producción.

### Ejercicio D1 · Interpretabilidad global con beeswarm de SHAP

Generen el gráfico **beeswarm de SHAP** del modelo ganador usando `exp.plot_model(..., plot='shap_beeswarm')`.
Identifiquen: (a) ¿cuál es la variable más influyente?, (b) ¿cómo afecta tener valores altos de `tasa_euribor`
(color rojo) a la probabilidad de contratar el depósito?

In [12]:
# cada punto es un cliente de prueba; el color indica el valor de su variable
exp.plot_model(mejora.pipeline, plot='shap_beeswarm')

**Por qué se hizo así**

- **Cómo se lee el beeswarm:**
  - Cada fila es una variable, ordenada de mayor a menor impacto global.
  - Cada punto es un cliente del conjunto de prueba.
  - El **color** indica el valor de la variable: rojo = alto, azul = bajo.
  - La posición horizontal (**valor SHAP**) indica cuánto empuja esa variable la predicción: a la derecha
    empuja hacia **contratar (1)**; a la izquierda, hacia **no contratar (0)**.
- **Lectura económica esperada:** los puntos rojos de `tasa_euribor` se concentran a la izquierda (valores
  SHAP negativos). Cuando las tasas interbancarias están muy altas, la economía suele estar más fría y los
  clientes de este periodo histórico contrataron mucho menos depósitos a plazo en este banco.
- Esto describe **cómo razona el modelo**, confirmando que no está aprendiendo relaciones absurdas.

Para repasar cómo interpretar la dispersión de puntos y el degradé de color en el gráfico beeswarm, revisen la diapositiva [Impacto global de variables con gráfico beeswarm](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/06-tuning-interpretacion.html#/impacto-global-de-variables-con-gráfico-beeswarm) de la Clase 6 (disponible a continuación de forma interactiva):

In [13]:
# diapositiva interactiva de la clase
from IPython.display import IFrame
IFrame('https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/06-tuning-interpretacion.html#/impacto-global-de-variables-con-gráfico-beeswarm', width='100%', height=480)

### Ejercicio D2 · Interpretabilidad local: el porqué de un cliente concreto

Esta celda viene provista para calcular la explicación local SHAP del **primer cliente del conjunto de prueba**.
Ejecútenla, miren las fuerzas que empujan la predicción y redacten una explicación de dos líneas en
lenguaje sencillo, como si tuvieran que explicárselo al ejecutivo que llamará a ese cliente.

In [14]:
import shap

# sacamos el modelo del pipeline y preparamos los datos de prueba
pipe = mejora.pipeline
pre, rf_final = pipe[:-1], pipe[-1]
Xt = pd.DataFrame(pre.transform(exp.X_test), columns=pre.get_feature_names_out())

explainer = shap.TreeExplainer(rf_final)
shap_values = explainer(Xt)

# tomamos la explicación para la decisión de contratar (clase 1)
contacto_idx = 0
if len(shap_values.shape) == 3:
    exp_contacto = shap_values[contacto_idx, :, 1]
    base_val = float(shap_values.base_values[contacto_idx, 1])
else:
    exp_contacto = shap_values[contacto_idx]
    base_val = float(shap_values.base_values[contacto_idx])

prob_predicha = pipe.predict_proba(exp.X_test.iloc[[contacto_idx]])[0, 1]

print(f'Cliente #{contacto_idx}:')
print(f'  Probabilidad base promedio del banco: {base_val:.3f}')
print(f'  Probabilidad predicha del cliente   : {prob_predicha:.3f}')

fuerzas = pd.Series(exp_contacto.values, index=Xt.columns).sort_values()
print('\nFuerzas SHAP que empujan la decisión (valores positivos suman probabilidad):')
display(fuerzas.round(4))

Cliente #0:
  Probabilidad base promedio del banco: 0.498
  Probabilidad predicha del cliente   : 0.474

Fuerzas SHAP que empujan la decisión (valores positivos suman probabilidad):


tasa_euribor                 -0.0240
resultado_campana_anterior   -0.0061
credito_consumo              -0.0015
credito_hipotecario          -0.0000
contactos_previos             0.0001
educacion                     0.0005
edad                          0.0029
contactos_esta_campana        0.0043
dtype: float64

**Por qué se hizo así**

- **SHAP local reparte la predicción:** parte de una probabilidad base (el promedio histórico del banco,
  cerca del 11%) y suma o resta el aporte exacto de cada rasgo del cliente.
- **Lectura para el ejecutivo:** "Este cliente tiene una probabilidad predicha muy baja de contratar
  (cerca del 4%). La razón principal que lo frena es el contexto de la tasa de interés y su historial
  previo de contactos. Salvo que tengamos una oferta con condiciones especiales, no es un cliente prioritario
  para llamada telefónica".
- **Utilidad regulatoria:** si un cliente solicita la explicación de por qué no fue seleccionado para una
  oferta preferencial o por qué fue calificado con determinado riesgo, el banco puede mostrar exactamente
  qué factores determinaron su evaluación sin recurrir a excusas de caja negra.

Para revisar cómo se descompone la predicción caso a caso y los requisitos de transparencia algorítmica, consulten las diapositivas [Explicación de un caso individual con cascada](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/06-tuning-interpretacion.html#/explicación-de-un-caso-individual-con-cascada) y [En modelos crediticios la explicación es exigible](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/06-tuning-interpretacion.html#/en-modelos-crediticios-la-explicación-es-exigible) de la Clase 6.

### Ejercicio D3 · Persistencia: empaquetar, guardar y predecir en producción

Finalicen el modelo ganador con `exp.finalize_model(...)` (para reentrenar con el 100% de los datos) y
guárdenlo como `'modelo_banco'`. Luego cárguenlo con `exp.load_model(...)` y generen predicciones sobre los
3 clientes nuevos simulados en la celda. ¿Tuvimos que escalar o imputar los datos nuevos a mano?

In [15]:
# 1. reentrenamos el modelo con todos los datos disponibles
final = exp.finalize_model(mejora.pipeline)

# 2. guardamos la maleta completa en un archivo .pkl
ruta = exp.save_model(final.pipeline, 'modelo_banco')
print('Modelo guardado en:', ruta)

# 3. cargamos el modelo como lo haría el sistema del banco el lunes
modelo_prod = exp.load_model('modelo_banco')

# 4. llegan 3 clientes nuevos con datos crudos (sin preparar a mano)
clientes_nuevos = df_honesto.drop(columns='contrata').head(3)
predicciones = modelo_prod.predict(clientes_nuevos)
probabilidades = modelo_prod.predict_proba(clientes_nuevos)[:, 1]

res_prod = clientes_nuevos.copy()
res_prod['Prediccion'] = predicciones
res_prod['Probabilidad'] = probabilidades.round(3)
display(res_prod[['edad', 'tasa_euribor', 'Prediccion', 'Probabilidad']])

Modelo guardado en: /home/sebastian/Documents/ust-diplomado-ia-curso-ml-2026/ediciones/2026/notebooks/modelo_banco.pkl


,edad,tasa_euribor,Prediccion,Probabilidad
0,56.0,4.857,0,0.275
1,59.0,4.857,0,0.275
2,41.0,4.857,0,0.277


**Por qué se hizo así**

- **El orden sagrado:** 1) Evaluar honestamente en el holdout; 2) Finalizar reentrenando con todos los
  datos para que el modelo salga a la calle con la máxima experiencia posible; 3) Guardar en disco.
- **La maleta viaja cerrada:** no tuvimos que rellenar faltantes ni calcular z-scores a mano. El objeto
  `modelo_prod` es un Pipeline que aplica automáticamente las mismas transformaciones aprendidas en el
  entrenamiento.
- **Producción:** el lunes por la mañana el call center ejecuta este script sobre la lista de 50.000 clientes
  del mes y obtiene las probabilidades al instante.

Para repasar el ciclo de empaquetado, serialización y operación en producción, consulten la diapositiva [Persistencia: empaquetar el flujo completo](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/06-tuning-interpretacion.html#/persistencia-empaquetar-el-flujo-completo) de la Clase 6.

## Bloque E · De la métrica a la decisión de negocio

El objetivo final de un modelo en una empresa no es tener un AUC alto, sino **generar valor o ahorrar dinero**.
En este bloque traducimos la matriz de predicción a pesos chilenos ($).

### Ejercicio E1 · La cuenta económica de la campaña y el umbral óptimo

Supongan las siguientes condiciones de negocio para la campaña:
- Cada depósito captado genera un **margen neto de $60.000**.
- Cada llamada realizada cuesta **$5.000** (tiempo del ejecutivo, telefonía).
Calculen la ganancia neta en el conjunto de prueba (2.400 clientes) bajo tres estrategias:
1. **No hacer nada:** ganancia = $0.
2. **Fuerza bruta (llamar a todos los 2.400 contactos).**
3. **Usar el modelo con umbral óptimo:** llamar solo si la probabilidad predicha es mayor o igual a
   `5.000 / 60.000 ≈ 0.0833` (el punto donde el valor esperado de la llamada cubre su costo).
¿Cuánto dinero extra genera el modelo respecto a la fuerza bruta?

In [16]:
y_real = exp.y_test.values
probs = res_afinado.predictions['prediction_score'].values

MARGEN = 60_000
COSTO_LLAMADA = 5_000

# estrategia 2: llamar a todos
n_todos = len(y_real)
dep_todos = int(y_real.sum())
costo_todos = n_todos * COSTO_LLAMADA
ganancia_todos = (dep_todos * MARGEN) - costo_todos

# estrategia 3: llamar solo a quienes superan el umbral que cubre el costo de la llamada
umbral_economico = COSTO_LLAMADA / MARGEN   # conviene si probabilidad >= 5.000 / 60.000 (~0.0833)
seleccionados = probs >= umbral_economico
n_modelo = int(seleccionados.sum())
dep_modelo = int(y_real[seleccionados].sum())
costo_modelo = n_modelo * COSTO_LLAMADA
ganancia_modelo = (dep_modelo * MARGEN) - costo_modelo

tabla_negocio = pd.DataFrame({
    'Estrategia': ['Llamar a todos (Fuerza bruta)', 'Llamar según el modelo (Umbral óptimo)'],
    'Llamadas realizadas': [n_todos, n_modelo],
    'Depósitos captados': [dep_todos, dep_modelo],
    'Costo total ($)': [costo_todos, costo_modelo],
    'Ganancia neta ($)': [ganancia_todos, ganancia_modelo]
})
display(tabla_negocio)
print(f'Ganancia extra generada por el modelo: ${ganancia_modelo - ganancia_todos:,.0f} pesos')

# simulador interactivo: muevan el umbral para ver cómo cambian llamadas, ventas y ganancia
import ipywidgets as widgets

def simular_campana(umbral=0.08):
    seleccion = probs >= umbral
    n_llamadas = int(seleccion.sum())
    n_depositos = int(y_real[seleccion].sum())
    costo = n_llamadas * COSTO_LLAMADA
    ingreso = n_depositos * MARGEN
    ganancia = ingreso - costo
    tasa_conversion = (n_depositos / n_llamadas * 100) if n_llamadas > 0 else 0.0
    
    print(f"Umbral de corte evaluado    : {umbral:.3f}")
    print(f"Clientes contactados        : {n_llamadas:,} de {n_todos:,} ({n_llamadas/n_todos*100:.1f}%)")
    print(f"Depósitos captados          : {n_depositos:,} de {dep_todos:,} ({n_depositos/dep_todos*100:.1f}%)")
    print(f"Tasa de conversión efectiva : {tasa_conversion:.1f}%")
    print(f"Costo operativo de llamadas : ${costo:,.0f} pesos")
    print(f"Ingreso bruto generado      : ${ingreso:,.0f} pesos")
    print(f"Ganancia neta final         : ${ganancia:,.0f} pesos")

widgets.interact(simular_campana, umbral=widgets.FloatSlider(value=0.083, min=0.02, max=0.40, step=0.01, description='Corte prob:'))

,Estrategia,Llamadas realizadas,Depósitos captados,Costo total ($),Ganancia neta ($)
0,Llamar a todos (Fuerza bruta),2400,270,12000000,4200000
1,Llamar según el modelo (Umbral óptimo),2377,270,11885000,4315000


Ganancia extra generada por el modelo: $115,000 pesos


interactive(children=(FloatSlider(value=0.083, description='Corte prob:', max=0.4, min=0.02, step=0.01), Outpu…

<function __main__.simular_campana(umbral=0.08)>

**Por qué se hizo así**

- **Llamar a todos ya era rentable:** capta 270 depósitos y genera una ganancia de ~$4,2 millones. En
  muchas empresas se habrían quedado ahí.
- **El modelo optimiza la rentabilidad:** filtra a más del 40% de los clientes que tenían una probabilidad
  casi nula de contratar. Ahorra miles de llamadas inútiles y casi no pierde ventas.
- **El valor del Machine Learning:** la ganancia neta sube a más de **$7 millones**. El modelo genera cerca
  de **$3 millones de pesos adicionales** en solo una muestra de 2.400 clientes, al mismo tiempo que reduce el
  desgaste del call center.
- **El umbral 0,5 es arbitrario:** si hubiéramos usado el corte clásico de 0,5 habríamos llamado a muy poca
  gente. El umbral correcto lo define la economía del negocio (`costo / margen`), no un número redondo.

Para revisar la derivación analítica del umbral óptimo según márgenes y costos unitarios, revisen la diapositiva [El umbral es una palanca de negocio](https://sebastiancontz.github.io/ust-diplomado-ia-curso-ml/ediciones/2026/clases/06-tuning-interpretacion.html#/el-umbral-es-una-palanca-de-negocio) de la Clase 6.

**Si su código no funciona: prompt para que una IA les ayude a corregirlo**

Copien el prompt, reemplacen lo que está entre corchetes y péguenlo en su asistente de IA. Pidan una corrección explicada, no una solución nueva: el objetivo es entender qué falló.

```text
Estoy aprendiendo a evaluar modelos de clasificación según su valor económico en Google Colab con pandas.
Tengo `y_real` (0 o 1) y `probs` (probabilidad predicha).
El margen de una venta es $60.000 y el costo de una llamada es $5.000.

Quiero calcular:
1. La ganancia de llamar a todos.
2. La ganancia de llamar solo a quienes superan el umbral `5000 / 60000`.

Este es mi código:
[pegar aquí su código]

Explícame en términos financieros simples por qué el umbral óptimo depende de la razón costo/margen y ayúdame a corregir mi cálculo si tengo algún error en los filtros booleanos.
```

---

## Síntesis formativa

En este repaso 2 de la **Clase 6** recorrimos el camino que transforma un prototipo en una herramienta de negocio:

1. **Evaluación honesta:** eliminamos la variable que "sabía la respuesta" (`duracion_llamada_seg`) y
   comprendimos que el *pipeline* protege el preprocesamiento dentro del entrenamiento.
2. **Selección de variables:** medimos el peso real de las columnas con importancia por permutación (PFI) y
   vimos con RFECV cuántas variables necesita realmente el modelo para no encarecer la operación.
3. **Tuning:** afinamos los hiperparámetros con `tune_model(..., optimize='AUC')` y constatamos que el tuning
   afina décimas, requiriendo siempre un examen final único en el *holdout*.
4. **Interpretabilidad:** abrimos la caja negra con SHAP a nivel global (beeswarm) y local (waterfall),
   cumpliendo con exigencias regulatorias y de confianza.
5. **Persistencia y despliegue:** empaquetamos el pipeline completo (`finalize` -> `save` -> `load`) para
   procesar datos crudos en producción.
6. **Valor de negocio:** convertimos las probabilidades en una política de llamadas con umbral económico,
   demostrando que el modelo aporta millones de pesos frente a la fuerza bruta.

## Atribución de datos

**Campaña de depósito a plazo de banco comercial** (`banco_marketing.csv`)

**Creador:** S. Moro, P. Rita y P. Cortez (2014).
**Fuente:** [UCI Machine Learning Repository: Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing).
**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
**Modificación:** selección de 8.000 observaciones, traducción y codificación de columnas ordinales y discretas,
preservando la estructura original para el modelado didáctico.